# paper-verdict-finetune -- Colab GPU runner

Before running: Runtime -> Change runtime type -> GPU.

This notebook is intentionally thin: it clones the repo, installs
requirements, and calls the scripts in `src/`. All logic lives there.

**Persistence model** (so a run can be launched, closed, and reviewed
later -- Colab's local disk does not survive a runtime reset):
- **Checkpoints + full per-run outputs** -> zipped at the end of the run
  and offered to you as a direct browser download (no third-party
  storage, no Google Drive). Save it wherever you keep model files;
  re-upload that zip in a later Colab session if you need the checkpoint
  again.
- **Lightweight run record** (config, metrics, confusion matrix -- no
  weights) -> committed and pushed back to this branch, so `RUNS.md` in
  the repo is always a current, append-only index of every run, readable
  from any future session with nothing more than `git clone`. GitHub
  holds the map of what exists and what it scored; the zip you downloaded
  holds the actual weights.

In [ ]:
!nvidia-smi

In [ ]:
# no external storage account needed -- checkpoints download straight to
# your browser at the end of the run (see the last cells of this notebook)
import os

In [ ]:
BRANCH = "verdict-finetune-architecture"
!git clone --branch $BRANCH --depth 1 https://github.com/AngryWatership/textor.git repo
%cd repo
!git config user.email "noreply@anthropic.com"
!git config user.name "Colab runner"

In [ ]:
!pip install -q -r requirements.txt

## Review past runs before launching a new one

`RUNS.md` came in with the clone -- it's the current index of everything
already tried. Look at this before picking the next config, so you're
not re-running something already done, and so you know what the current
best is to try to beat.

In [ ]:
print(open("RUNS.md").read())

In [ ]:
# sanity-check the split before committing GPU time to training
!python src/data.py --data data/papers.jsonl --split-out outputs/split.json

In [ ]:
import sys
sys.path.insert(0, "src")
from runlog import new_run_id

BASE_MODEL = "allenai/specter2_base"  # or sentence-transformers/all-MiniLM-L6-v2, allenai/scibert_scivocab_uncased
FINETUNE_MODE = "partial"  # or "full"

RUN_ID = new_run_id(BASE_MODEL, FINETUNE_MODE)
print("RUN_ID:", RUN_ID)
os.environ["RUN_ID"] = RUN_ID
os.environ["BASE_MODEL"] = BASE_MODEL
os.environ["FINETUNE_MODE"] = FINETUNE_MODE

In [ ]:
!cd src && python train.py \
    --base-model $BASE_MODEL \
    --finetune-mode $FINETUNE_MODE \
    --run-id $RUN_ID \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs \
    --repo-root ..

In [ ]:
!cd src && python evaluate.py \
    --run-id $RUN_ID \
    --base-model $BASE_MODEL \
    --finetune-mode $FINETUNE_MODE \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs \
    --repo-root ..

In [ ]:
import json
report = json.load(open(f"outputs/{RUN_ID}/eval_report.json"))
print("confusion matrix (rows=true, cols=pred), labels:", report["confusion_matrix_labels"])
for row in report["confusion_matrix"]:
    print(row)
print("test silhouette:", report["test_silhouette_euclidean"])
print("test knn10 purity:", report["test_knn10_purity"])
print("per-class purity:", report["test_knn10_purity_per_class"])
print("\ncompare against docs/prior_results.md and RUNS.md for every other run so far.")

## Persist: zip the full run (incl. weights) for download, lightweight record to GitHub

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(f"outputs/{RUN_ID}", "zip", f"outputs/{RUN_ID}")
print(f"zipped full run (including best_model.pt) to {zip_path}")
print("a download prompt will appear below -- save it somewhere you keep model files.")
files.download(zip_path)

# nothing external to record a URL for -- note in the log that the
# checkpoint only exists as a browser download from this run, not a
# re-fetchable remote path
from runlog import upsert_run
upsert_run(".", RUN_ID, {"checkpoint_path": f"downloaded as {RUN_ID}.zip (no remote copy)"})

In [ ]:
# push the lightweight record (RUNS.md, runs_log.jsonl, this run's config/
# train_summary/eval_report -- NOT best_model.pt, that was already handed
# to you as a zip download above) back to the branch, so it's reviewable
# from any future session with nothing more than a git clone.
!git add RUNS.md runs_log.jsonl outputs/$RUN_ID/config.json outputs/$RUN_ID/train_summary.json outputs/$RUN_ID/eval_report.json
!git commit -m "Log run $RUN_ID" || echo "nothing new to commit"
!git push origin $BRANCH

## To resume / review in a later session

- `git clone` this branch -> `RUNS.md` shows every run and its metrics,
  no account or token needed to see *what happened*.
- The checkpoint itself only exists in the zip you downloaded when the
  run finished (`RUNS.md` notes this in the `checkpoint` column). To use
  it again, upload that zip into a new Colab session, unzip it, and point
  `evaluate.py --checkpoint` at the extracted `best_model.pt`.